# BERT Ablation — Training Data Variants (Point 7)

Fine-tunes **BERT** on four training-set variants and compares their
performance on the same held-out test set.

Variants (Point 7, prof. feedback):

1. **A — real only**: `edit_type ∈ {original, relabelled_only}`
2. **B — real + strengthened**: adds `strengthened`
3. **C — all edited** (baseline, matches the main fine-tune): adds `counterfactual`
4. **D — all except Gemini CFs**: removes `edit_type == 'counterfactual' AND source_canonical == 'gemini'`

For each variant × seed reports **Accuracy / F1 / AUC** on the main test,
**LOSO mean F1** and **LOSO Gemini F1**.

**Compute note**: 4 variants × len(SEEDS) × (1 main + 3 LOSO) fine-tunes.
With 3 seeds that's 48 fine-tunes (~3–5 min each on a T4). The cell saves a
checkpoint per run in OUT_DIR, so `USE_SAVED_MODELS = True` will resume a
previous run that got interrupted.


In [ ]:
!pip -q install --upgrade transformers torch torchvision scikit-learn pandas numpy

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
)
from sklearn.model_selection import GroupShuffleSplit

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to your Drive folder
ROOT = Path('/content/drive/MyDrive/project-colab')
ROOT.mkdir(parents=True, exist_ok=True)

DATA_JSON = ROOT / 'bias_sentences_v9.json'
OUT_DIR   = ROOT / 'bert_ablation_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_JSON:', DATA_JSON)
print('OUT_DIR  :', OUT_DIR)


Mounted at /content/drive
DATA_JSON: /content/drive/MyDrive/project-colab/bias_sentences_v9.json
OUT_DIR  : /content/drive/MyDrive/project-colab/bert_ablation_outputs


In [3]:
with open(DATA_JSON, encoding='utf-8') as f:
    raw = json.load(f)

df_sentences = pd.DataFrame(raw['entries']).copy()
df_sentences['label'] = df_sentences['has_bias'].astype(int)

SOURCE_CANONICAL = {
    'biased_corpus_only': 'biased-corpus',
    'biased_corpus_v2':   'biased-corpus',
    'gemini_only':        'gemini',
    'gemini_only_v2':     'gemini',
    'gus_only':           'gus-dataset',
    'gus_only_v2':        'gus-dataset',
}
df_sentences['source_canonical'] = (
    df_sentences['source'].map(SOURCE_CANONICAL).fillna(df_sentences['source'])
)

df = df_sentences.copy()  # fine-tuned BERT/GPT-2 only needs text + label + metadata
y       = df['label'].astype(int)
texts   = df['text'].fillna('').values
sources = df['source_canonical'].values
unique_sources = np.array(sorted(np.unique(sources)))

print('N:', len(df))
print('Label dist:', y.value_counts().to_dict())
print('Sources:', pd.Series(sources).value_counts().to_dict())
print('Roles:', df['role'].value_counts().to_dict())
print('Edit types:', df['edit_type'].value_counts().to_dict())


N: 10304
Label dist: {1: 5497, 0: 4807}
Sources: {'gemini': 4440, 'biased-corpus': 3235, 'gus-dataset': 2629}
Roles: {'original': 6457, 'counterfactual': 3847}
Edit types: {'original': 5094, 'counterfactual': 2926, 'strengthened': 1349, 'relabelled_only': 935}


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# ─── Variant / seed configuration ───
SEEDS = [1, 2, 3, 4, 5]  # full run is [1,2,3,4,5]; use [1,2,3] or [1] for a pilot

# The 4 training-data variants (Point 7, prof. feedback)
VARIANTS = {
    'A_real_only':       lambda r: r['edit_type'].isin(['original', 'relabelled_only']),
    'B_real_plus_str':   lambda r: r['edit_type'].isin(['original', 'relabelled_only', 'strengthened']),
    'C_all_edited':      lambda r: r['edit_type'].notna() | r['edit_type'].isna(),
    'D_no_gemini_cfs':   lambda r: ~((r['edit_type'] == 'counterfactual') & (r['source_canonical'] == 'gemini')),
}
# Restrict to a subset if you want to iterate faster, e.g. ['A_real_only', 'D_no_gemini_cfs']
VARIANTS_TO_RUN = list(VARIANTS.keys())

# ─── Training hyper-parameters (same as baseline) ───
MODEL_ID   = 'bert-base-uncased'
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 20
LR         = 2e-5

RUN_LOSO        = True   # also fine-tune LOSO models per (variant, seed, held-out source)
USE_SAVED_MODELS = True  # skip retraining if the checkpoint already exists on Drive
SAVE_MODELS      = True  # write checkpoints after training (so the run is resumable)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'\nModel: {MODEL_ID} (BERT)')
print(f'Seeds: {SEEDS}')
print(f'Variants to run: {VARIANTS_TO_RUN}')


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Model: bert-base-uncased (BERT)
Seeds: [1, 2, 3, 4, 5]
Variants to run: ['A_real_only', 'B_real_plus_str', 'C_all_edited', 'D_no_gemini_cfs']


In [5]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def encode_texts(text_array, tok, max_len):
    enc = tok(
        list(text_array),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']


def train_classifier(X_ids, X_mask, y_train,
                     X_ids_val, X_mask_val, y_val,
                     seed, epochs=EPOCHS, lr=LR, patience=3):
    set_all_seeds(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=2
    )
    if hasattr(model.config, 'pad_token_id') and model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    model.to(device)

    dataset = TensorDataset(
        X_ids.to(device),
        X_mask.to(device),
        torch.tensor(y_train, dtype=torch.long).to(device),
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    val_dataset = TensorDataset(
        X_ids_val.to(device),
        X_mask_val.to(device),
        torch.tensor(y_val, dtype=torch.long).to(device),
    )
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    best_val_loss = float('inf')
    best_state = None
    bad_epochs = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for ids, mask, labels in loader:
            optimizer.zero_grad()
            outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for ids, mask, labels in val_loader:
                outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
                val_loss += outputs.loss.item()
        val_loss /= max(len(val_loader), 1)

        print(f'      Epoch {epoch+1}/{epochs}  loss={total_loss/len(loader):.4f}  val_loss={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f'      Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    return model


def predict_probs(model, X_ids, X_mask):
    model.eval()
    dataset = TensorDataset(X_ids.to(device), X_mask.to(device))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    all_probs = []
    with torch.no_grad():
        for ids, mask in loader:
            outputs = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs)


def ckpt_main(variant, seed):
    return OUT_DIR / f'{variant}_seed_{seed}_main'


def ckpt_loso(variant, seed, src):
    safe_src = str(src).replace('/', '_')
    return OUT_DIR / f'{variant}_seed_{seed}_loso_{safe_src}'


In [6]:
def pair_aware_split(seed, test_size=0.25, val_size=0.20):
    """
    Builds a per-seed pair-aware split identical to bert_attention_pipeline.ipynb Cell 44:
      1. GSS on non-CF rows, grouped by pair_id (test_size=0.25)
      2. Keep only edit_type=='original' in test; move rewritten to train
      3. Attach CFs whose pair_id is already in train
    Then splits train into train_fit (80%) / val (20%) for early stopping.
    """
    is_cf = (df['role'] == 'counterfactual').values
    non_cf = np.where(~is_cf)[0]
    cf_all = np.where(is_cf)[0]

    nc_pids = df.iloc[non_cf]['pair_id'].copy()
    m_na = nc_pids.isna()
    nc_pids[m_na] = ['unpaired_' + str(i) for i in range(m_na.sum())]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tr_l, te_l = next(gss.split(non_cf, y.iloc[non_cf], groups=nc_pids.values))
    tr_noncf = non_cf[tr_l]
    te_raw   = non_cf[te_l]

    te_et = df.iloc[te_raw]['edit_type'].values
    keep = np.isin(te_et, ['original'])
    test_idx_s  = te_raw[keep]
    tr_noncf    = np.concatenate([tr_noncf, te_raw[~keep]])

    tr_pairs = set(df.iloc[tr_noncf]['pair_id'].dropna())
    cf_in_tr = cf_all[df.iloc[cf_all]['pair_id'].isin(tr_pairs).values]
    train_idx_s = np.concatenate([tr_noncf, cf_in_tr])

    # Train-fit / val split (for early stopping)
    tr_pids = df.iloc[train_idx_s]['pair_id'].copy()
    m_na2 = tr_pids.isna()
    tr_pids[m_na2] = ['unpaired_' + str(i) for i in range(m_na2.sum())]
    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    fit_loc, val_loc = next(gss_val.split(train_idx_s, y.iloc[train_idx_s], groups=tr_pids.values))
    train_fit_idx_s = train_idx_s[fit_loc]
    val_idx_s       = train_idx_s[val_loc]

    return train_idx_s, train_fit_idx_s, val_idx_s, test_idx_s


In [7]:
import gc

abl_main_rows = []
abl_loso_rows = []

# Re-encode per seed because the test/val set changes with the seed
for seed in SEEDS:
    print(f'\n{"#"*72}')
    print(f'# SEED {seed}')
    print(f'{"#"*72}')

    train_idx_s, train_fit_idx_s, val_idx_s, test_idx_s = pair_aware_split(seed)
    print(f'  split: train={len(train_idx_s)}  fit={len(train_fit_idx_s)}  val={len(val_idx_s)}  test={len(test_idx_s)}')

    # Encode val/test once per seed (same across variants)
    ids_val, mask_val = encode_texts(texts[val_idx_s], tokenizer, MAX_LEN)
    ids_te,  mask_te  = encode_texts(texts[test_idx_s], tokenizer, MAX_LEN)
    y_val_arr = y.iloc[val_idx_s].values
    y_te_arr  = y.iloc[test_idx_s].values

    for vname in VARIANTS_TO_RUN:
        vfn = VARIANTS[vname]
        print(f'\n  === Variant: {vname} (seed {seed}) ===')

        # ── Apply variant filter to train_fit + val  ──
        fit_sub = df.iloc[train_fit_idx_s]
        keep_fit = vfn(fit_sub).values
        tr_fit_v = train_fit_idx_s[keep_fit]

        val_sub = df.iloc[val_idx_s]
        keep_val = vfn(val_sub).values
        val_v = val_idx_s[keep_val]

        ids_tr, mask_tr = encode_texts(texts[tr_fit_v], tokenizer, MAX_LEN)
        y_tr_arr = y.iloc[tr_fit_v].values

        if len(val_v) >= BATCH_SIZE:
            ids_val_v, mask_val_v = encode_texts(texts[val_v], tokenizer, MAX_LEN)
            y_val_v = y.iloc[val_v].values
        else:
            # tiny val set (edge case) — fall back to the full val
            ids_val_v, mask_val_v, y_val_v = ids_val, mask_val, y_val_arr

        print(f'    train_fit={len(tr_fit_v)}  val={len(val_v)}')
        print(f'    label dist train: {pd.Series(y_tr_arr).value_counts().to_dict()}')

        main_dir = ckpt_main(vname, seed)
        if USE_SAVED_MODELS and main_dir.exists():
            print(f'    [load] {main_dir.name}')
            model = AutoModelForSequenceClassification.from_pretrained(main_dir).to(device)
        else:
            model = train_classifier(
                ids_tr, mask_tr, y_tr_arr,
                ids_val_v, mask_val_v, y_val_v,
                seed=seed,
            )
            if SAVE_MODELS:
                main_dir.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(main_dir)
                tokenizer.save_pretrained(main_dir)
                print(f'    [save] {main_dir.name}')

        probs_te = predict_probs(model, ids_te, mask_te)
        preds_te = (probs_te >= 0.5).astype(int)

        row = {
            'variant': vname, 'seed': seed,
            'n_train_fit': int(len(tr_fit_v)),
            'accuracy': accuracy_score(y_te_arr, preds_te),
            'f1':       f1_score(y_te_arr, preds_te, zero_division=0),
            'precision': precision_score(y_te_arr, preds_te, zero_division=0),
            'recall':    recall_score(y_te_arr, preds_te, zero_division=0),
            'auc':      roc_auc_score(y_te_arr, probs_te),
        }
        print(f'    MAIN  acc={row["accuracy"]:.4f}  f1={row["f1"]:.4f}  auc={row["auc"]:.4f}')

        loso_f1s = {}
        if RUN_LOSO:
            for test_src in unique_sources:
                tmask = sources == test_src
                tr_full = np.where(~tmask)[0]
                te_l    = np.where(tmask)[0]

                keep_l = vfn(df.iloc[tr_full]).values
                tr_l   = tr_full[keep_l]
                if len(tr_l) == 0 or y.iloc[tr_l].nunique() < 2:
                    loso_f1s[test_src] = np.nan
                    continue

                # LOSO val: 20% of variant-filtered train
                pids_l = df.iloc[tr_l]['pair_id'].copy()
                m_na_l = pids_l.isna()
                pids_l[m_na_l] = ['unpaired_' + str(i) for i in range(m_na_l.sum())]
                gss_lv = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
                tr_lv, val_lv = next(gss_lv.split(tr_l, y.iloc[tr_l], groups=pids_l.values))
                tr_l_fit = tr_l[tr_lv]
                val_l    = tr_l[val_lv]

                ids_tr_l, mask_tr_l   = encode_texts(texts[tr_l_fit], tokenizer, MAX_LEN)
                ids_val_l, mask_val_l = encode_texts(texts[val_l],   tokenizer, MAX_LEN)
                ids_te_l, mask_te_l   = encode_texts(texts[te_l],    tokenizer, MAX_LEN)

                loso_dir = ckpt_loso(vname, seed, test_src)
                if USE_SAVED_MODELS and loso_dir.exists():
                    print(f'      [LOSO {test_src}] load {loso_dir.name}')
                    model_l = AutoModelForSequenceClassification.from_pretrained(loso_dir).to(device)
                else:
                    print(f'      [LOSO {test_src}] training (n_tr_fit={len(tr_l_fit)})')
                    model_l = train_classifier(
                        ids_tr_l, mask_tr_l, y.iloc[tr_l_fit].values,
                        ids_val_l, mask_val_l, y.iloc[val_l].values,
                        seed=seed,
                    )
                    if SAVE_MODELS:
                        loso_dir.mkdir(parents=True, exist_ok=True)
                        model_l.save_pretrained(loso_dir)
                        tokenizer.save_pretrained(loso_dir)

                probs_l = predict_probs(model_l, ids_te_l, mask_te_l)
                preds_l = (probs_l >= 0.5).astype(int)
                y_te_l  = y.iloc[te_l].values

                f1_l = f1_score(y_te_l, preds_l, zero_division=0)
                loso_f1s[test_src] = f1_l
                try:
                    auc_l = roc_auc_score(y_te_l, probs_l)
                except ValueError:
                    auc_l = np.nan

                abl_loso_rows.append({
                    'variant': vname, 'seed': seed, 'source': test_src,
                    'accuracy': accuracy_score(y_te_l, preds_l),
                    'f1': f1_l,
                    'precision': precision_score(y_te_l, preds_l, zero_division=0),
                    'recall':    recall_score(y_te_l, preds_l, zero_division=0),
                    'auc': auc_l,
                    'n_train': int(len(tr_l_fit)),
                })
                print(f'      [LOSO {test_src}] f1={f1_l:.4f}  auc={auc_l if auc_l==auc_l else float("nan"):.4f}')

                del model_l
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

        row['loso_mean_f1']   = float(np.nanmean(list(loso_f1s.values()))) if loso_f1s else np.nan
        row['loso_gemini_f1'] = float(loso_f1s.get('gemini', np.nan))
        abl_main_rows.append(row)

        # Persist incrementally in case Colab disconnects
        pd.DataFrame(abl_main_rows).to_csv(OUT_DIR / 'df_bert_ablation_main.csv', index=False)
        pd.DataFrame(abl_loso_rows).to_csv(OUT_DIR / 'df_bert_ablation_loso.csv', index=False)

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

df_abl_main = pd.DataFrame(abl_main_rows)
df_abl_loso = pd.DataFrame(abl_loso_rows)
print(f'\n[done] {len(df_abl_main)} rows in df_abl_main  |  {len(df_abl_loso)} rows in df_abl_loso')



########################################################################
# SEED 1
########################################################################
  split: train=8243  fit=6589  val=1654  test=1281

  === Variant: A_real_only (seed 1) ===
    train_fit=3709  val=949
    label dist train: {0: 2021, 1: 1688}
    [load] A_real_only_seed_1_main


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

    MAIN  acc=0.9571  f1=0.9478  auc=0.9891
      [LOSO biased-corpus] load A_real_only_seed_1_loso_biased-corpus


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

      [LOSO biased-corpus] f1=0.9186  auc=0.9695
      [LOSO gemini] load A_real_only_seed_1_loso_gemini


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


      Epoch 1/20  loss=0.5305  val_loss=0.2480
      Epoch 2/20  loss=0.1786  val_loss=0.1451
      Epoch 3/20  loss=0.0803  val_loss=0.1619
      Epoch 4/20  loss=0.0351  val_loss=0.2131
      Epoch 5/20  loss=0.0189  val_loss=0.2662
      Early stopping at epoch 5 (best val_loss=0.1451)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

      [LOSO gus-dataset] f1=0.9235  auc=0.9817

[done] 20 rows in df_abl_main  |  60 rows in df_abl_loso


In [8]:
from IPython.display import display

print(f'\n{"="*80}')
print(f'Training-Data Ablation Summary — {len(SEEDS)} seeds')
print(f'{"="*80}')

_metric_cols = ['accuracy', 'f1', 'auc', 'loso_mean_f1', 'loso_gemini_f1']
summary = (
    df_abl_main.groupby('variant')[_metric_cols]
    .agg(['mean', 'std']).round(4)
)

def _fmt(m, s):
    if m != m: return '—'
    if s != s: return f'{m:.4f}'
    return f'{m:.4f} ± {s:.4f}'

rows = []
for v, r in summary.iterrows():
    n_train_mean = df_abl_main.loc[df_abl_main['variant'] == v, 'n_train_fit'].mean()
    rows.append({
        'Variant':        v,
        'n_train (avg)':  f'{n_train_mean:.0f}',
        'Accuracy':       _fmt(r[('accuracy',       'mean')], r[('accuracy',       'std')]),
        'F1':             _fmt(r[('f1',             'mean')], r[('f1',             'std')]),
        'AUC':            _fmt(r[('auc',            'mean')], r[('auc',            'std')]),
        'LOSO mean F1':   _fmt(r[('loso_mean_f1',   'mean')], r[('loso_mean_f1',   'std')]),
        'LOSO Gemini F1': _fmt(r[('loso_gemini_f1', 'mean')], r[('loso_gemini_f1', 'std')]),
    })

df_abl_summary = pd.DataFrame(rows)
display(df_abl_summary)

# LOSO breakdown per source
if not df_abl_loso.empty:
    print('\nLOSO F1 by variant × source (mean ± std):')
    piv = (df_abl_loso.groupby(['variant', 'source'])['f1']
           .agg(['mean', 'std']).round(4))
    rows2 = []
    for (v, src), r in piv.iterrows():
        rows2.append({'Variant': v, 'Source': src,
                      'F1': f"{r['mean']:.4f} ± {r['std']:.4f}"})
    display(pd.DataFrame(rows2).pivot(index='Variant', columns='Source', values='F1'))

# Save final CSVs
df_abl_main.to_csv(OUT_DIR / 'df_bert_ablation_main.csv', index=False)
df_abl_loso.to_csv(OUT_DIR / 'df_bert_ablation_loso.csv', index=False)
print('\nSaved:')
print(' -', OUT_DIR / 'df_bert_ablation_main.csv')
print(' -', OUT_DIR / 'df_bert_ablation_loso.csv')



Training-Data Ablation Summary — 5 seeds


,Variant,n_train (avg),Accuracy,F1,AUC,LOSO mean F1,LOSO Gemini F1
0,A_real_only,3738,0.9599 ± 0.0086,0.9526 ± 0.0093,0.9927 ± 0.0025,0.8892 ± 0.0137,0.8271 ± 0.0461
1,B_real_plus_str,4721,0.9627 ± 0.0038,0.9559 ± 0.0056,0.9935 ± 0.0022,0.8973 ± 0.0091,0.8391 ± 0.0265
2,C_all_edited,6577,0.9607 ± 0.0061,0.9539 ± 0.0066,0.9932 ± 0.0015,0.9072 ± 0.0129,0.8659 ± 0.0293
3,D_no_gemini_cfs,5505,0.9636 ± 0.0121,0.9566 ± 0.0136,0.9930 ± 0.0021,0.9119 ± 0.0101,0.8659 ± 0.0293



LOSO F1 by variant × source (mean ± std):


Source,biased-corpus,gemini,gus-dataset
Variant,,,
A_real_only,0.9274 ± 0.0074,0.8271 ± 0.0461,0.9132 ± 0.0128
B_real_plus_str,0.9376 ± 0.0046,0.8391 ± 0.0265,0.9151 ± 0.0119
C_all_edited,0.9384 ± 0.0059,0.8659 ± 0.0293,0.9172 ± 0.0188
D_no_gemini_cfs,0.9396 ± 0.0029,0.8659 ± 0.0293,0.9301 ± 0.0049



Saved:
 - /content/drive/MyDrive/project-colab/bert_ablation_outputs/df_bert_ablation_main.csv
 - /content/drive/MyDrive/project-colab/bert_ablation_outputs/df_bert_ablation_loso.csv


In [11]:
# LOSO breakdown per source — accuracy, F1, AUC (variant × metric as rows)
if not df_abl_loso.empty:
    print('\nLOSO by variant × metric × source (mean ± std):')
    metrics = ['accuracy', 'f1', 'auc']
    agg = (df_abl_loso.groupby(['variant', 'source'])[metrics]
           .agg(['mean', 'std']).round(4))

    means = agg.xs('mean', axis=1, level=1)
    stds  = agg.xs('std',  axis=1, level=1)

    # "mean ± std" per cell
    fmt = means.copy().astype(object)
    for m in metrics:
        fmt[m] = [f'{mu:.4f} ± {sd:.4f}' for mu, sd in zip(means[m], stds[m])]

    # Reshape: rows=(variant, metric), columns=source
    out = (fmt.stack()                          # index: (variant, source, metric)
              .unstack('source')                # index: (variant, metric)
              .reindex(metrics, level=1))       # ordem accuracy → f1 → auc
    out.index.names = ['Variant', 'Metric']
    display(out)



LOSO by variant × metric × source (mean ± std):


source                      biased-corpus           gemini      gus-dataset
Variant         Metric                                                     
A_real_only     accuracy  0.9041 ± 0.0107  0.8441 ± 0.0328  0.9121 ± 0.0145
                f1        0.9274 ± 0.0074  0.8271 ± 0.0461  0.9132 ± 0.0128
                auc       0.9725 ± 0.0040  0.9087 ± 0.0208  0.9775 ± 0.0056
B_real_plus_str accuracy  0.9188 ± 0.0064  0.8524 ± 0.0209  0.9146 ± 0.0149
                f1        0.9376 ± 0.0046  0.8391 ± 0.0265  0.9151 ± 0.0119
                auc       0.9768 ± 0.0044  0.9120 ± 0.0197  0.9797 ± 0.0029
C_all_edited    accuracy  0.9209 ± 0.0062  0.8760 ± 0.0226  0.9162 ± 0.0223
                f1        0.9384 ± 0.0059  0.8659 ± 0.0293  0.9172 ± 0.0188
                auc       0.9777 ± 0.0017  0.9426 ± 0.0084  0.9799 ± 0.0020
D_no_gemini_cfs accuracy  0.9221 ± 0.0040  0.8760 ± 0.0226  0.9308 ± 0.0052
                f1        0.9396 ± 0.0029  0.8659 ± 0.0293  0.9301 ± 0.0049
                auc       0.9763 ± 0.0063  0.9426 ± 0.0084  0.9824 ± 0.0024